# Clasificación de noticias (AG News) — 4 modelos en PyTorch

**Proyecto final · Deep Learning**

Este notebook entrena **cuatro modelos** sobre AG News y guarda **checkpoints después de cada época**,
de modo que si Kaggle corta la sesión a mitad del entrenamiento, al volver a ejecutar se **reanuda
automáticamente** desde la última época completada.

Modelos:

1. **BiLSTM + atención entrenada desde cero** (red recurrente, sin pesos preentrenados) — *modelo base 1*.
2. **TF-IDF + Regresión Softmax** (regresión logística multinomial en PyTorch) — *modelo base 2, versión final*.
3. **DistilBERT ajustado (fine-tuning)** — *modelo estado del arte*.
4. **NewsHybridNet** — arquitectura propia de doble rama (semántica secuencial + léxica de subpalabras) con fusión por compuerta.

### Qué se guarda por modelo (requisito obligatorio)
Dentro de `OUTPUT_DIR/checkpoints/<modelo>/`:
- `last.pt` — se sobrescribe tras **cada época** (modelo, optimizador, scheduler, época, métricas, estados RNG).
- `best.pt` — mejor macro-F1 de validación visto hasta el momento.
- `history.json` — curvas de loss y métricas por época.
- Artefactos auxiliares (vocabulario, vectorizador TF-IDF, tokenizer, embeddings) para poder servir el modelo luego vía API.

La reanudación es automática: si existe `last.pt`, el entrenamiento continúa desde la época siguiente.


## 0. Instalación de dependencias

En Kaggle, `torch`, `scikit-learn`, `datasets` y `transformers` suelen venir preinstalados.
Esta celda solo asegura versiones compatibles; si ya están, no hace nada costoso.
Activa internet en el panel de Kaggle (**Settings → Internet: On**) para descargar el dataset y DistilBERT.

In [ ]:
# Descomenta si hace falta instalar/actualizar en tu entorno de Kaggle.
# !pip install -q -U "transformers>=4.40" "datasets>=2.19" "scikit-learn>=1.3"

import importlib, sys
for pkg in ["torch", "sklearn", "datasets", "transformers"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:14s} OK  ->", getattr(m, "__version__", "?"))
    except Exception as e:
        print(f"{pkg:14s} FALTA ->", e)


## 1. Configuración global, semillas y rutas de checkpoint

`OUTPUT_DIR` apunta a `/kaggle/working` cuando se ejecuta en Kaggle (ese directorio se conserva como
*output* del notebook). Si no existe, cae a una carpeta local `./outputs`, así el notebook también
corre fuera de Kaggle.

In [ ]:
import os, json, time, random, math, re
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Reproducibilidad ----
SEED = 30

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# ---- Dispositivo ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---- Rutas ----
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working")
else:
    OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_ROOT = OUTPUT_DIR / "checkpoints"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
NUM_CLASSES = len(CLASS_NAMES)

print("OUTPUT_DIR :", OUTPUT_DIR)
print("CKPT_ROOT  :", CKPT_ROOT)


## 2. Carga del dataset AG News y partición train/val/test

Se carga `fancyzhx/ag_news` desde Hugging Face `datasets`. Se separa un **10 % estratificado** del
entrenamiento como validación (semilla fija), dejando ~108.000 train / ~12.000 val, y se reserva el
test oficial (7.600) para el final, tal como describe el anteproyecto.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

raw = load_dataset("fancyzhx/ag_news")
print(raw)

train_text_all = [t for t in raw["train"]["text"]]
train_label_all = [int(y) for y in raw["train"]["label"]]
test_text = [t for t in raw["test"]["text"]]
test_label = [int(y) for y in raw["test"]["label"]]

train_text, val_text, train_label, val_label = train_test_split(
    train_text_all, train_label_all,
    test_size=0.10, random_state=SEED, stratify=train_label_all,
)

print(f"train: {len(train_text):>6}  val: {len(val_text):>6}  test: {len(test_text):>6}")
print("Ejemplo:", train_text[0][:120], "-> clase", CLASS_NAMES[train_label[0]])


## 3. Utilidades de checkpoint y métricas (compartidas por los 4 modelos)

El corazón del requisito: `save_checkpoint` escribe de forma **atómica** (archivo temporal + `os.replace`)
para que un corte justo durante el guardado no deje un checkpoint corrupto. `load_checkpoint` restaura
modelo, optimizador, scheduler, época y estados de los generadores aleatorios.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }

def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def set_rng_state(state):
    try:
        random.setstate(state["python"])
        np.random.set_state(state["numpy"])
        torch.set_rng_state(state["torch"])
        if state.get("cuda") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(state["cuda"])
    except Exception as e:
        print("Aviso: no se pudo restaurar RNG por completo:", e)

def atomic_save(obj, path: Path):
    """Guarda de forma atómica para no corromper el archivo si la sesión se corta."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)  # reemplazo atómico en el mismo sistema de archivos

def save_checkpoint(ckpt_dir: Path, model, optimizer, scheduler, epoch,
                    best_metric, history, extra=None, is_best=False):
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": best_metric,
        "rng_state": rng_state(),
        "extra": extra or {},
    }
    atomic_save(payload, ckpt_dir / "last.pt")
    with open(ckpt_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    if is_best:
        atomic_save(payload, ckpt_dir / "best.pt")
    print(f"  [checkpoint] epoca {epoch} guardada en {ckpt_dir}"
          + ("  (NUEVO MEJOR)" if is_best else ""))

def load_checkpoint(ckpt_dir: Path, model, optimizer=None, scheduler=None, map_location=None):
    """Devuelve (start_epoch, best_metric, history). Si no hay checkpoint, (0, -inf, [])."""
    ckpt_dir = Path(ckpt_dir)
    last = ckpt_dir / "last.pt"
    hist_path = ckpt_dir / "history.json"
    history = []
    if hist_path.exists():
        try:
            history = json.load(open(hist_path))
        except Exception:
            history = []
    if not last.exists():
        print(f"  [reanudar] no hay checkpoint en {ckpt_dir}; empezando desde cero.")
        return 0, float("-inf"), history
    ckpt = torch.load(last, map_location=map_location or DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and ckpt.get("optimizer_state"):
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt.get("scheduler_state"):
        scheduler.load_state_dict(ckpt["scheduler_state"])
    set_rng_state(ckpt["rng_state"])
    start_epoch = int(ckpt["epoch"])  # ya completada -> seguimos en la siguiente
    best_metric = ckpt.get("best_metric", float("-inf"))
    print(f"  [reanudar] checkpoint encontrado: epoca {start_epoch} completada, "
          f"best_macro_f1={best_metric:.4f}. Continuando desde la epoca {start_epoch+1}.")
    return start_epoch, best_metric, history


## 4. Modelo 1 — BiLSTM + atención (entrenado desde cero)

Red recurrente **sin ningún peso preentrenado**: la capa de embedding se inicializa al azar y se
aprende durante el entrenamiento. Arquitectura: `Embedding → BiLSTM → atención aditiva → MLP → softmax`.

### 4.1 Vocabulario y tokenización por palabras

In [ ]:
from collections import Counter

PAD, UNK = "<pad>", "<unk>"
MAX_LEN = 80          # longitud de secuencia en palabras (como en el anteproyecto)
MIN_FREQ = 2
MAX_VOCAB = 40000

_token_re = re.compile(r"[a-z0-9]+|[^\sa-z0-9]")

def tokenize(text):
    return _token_re.findall(text.lower())

def build_vocab(texts, min_freq=MIN_FREQ, max_vocab=MAX_VOCAB):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    itos = [PAD, UNK]
    for tok, c in counter.most_common():
        if c < min_freq or len(itos) >= max_vocab:
            break
        itos.append(tok)
    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos

# El vocabulario se construye SOLO con el conjunto de entrenamiento.
stoi, itos = build_vocab(train_text)
PAD_IDX = stoi[PAD]
print("Tamano de vocabulario:", len(itos))

def encode(text, max_len=MAX_LEN):
    ids = [stoi.get(tok, stoi[UNK]) for tok in tokenize(text)][:max_len]
    length = len(ids)
    if length < max_len:
        ids = ids + [PAD_IDX] * (max_len - length)
    return ids, max(length, 1)

# Guardar el vocabulario para reconstruir el modelo al servir la API.
vocab_path = CKPT_ROOT / "rnn_scratch"
vocab_path.mkdir(parents=True, exist_ok=True)
with open(vocab_path / "vocab.json", "w") as f:
    json.dump({"itos": itos, "max_len": MAX_LEN, "pad_idx": PAD_IDX}, f)
print("Vocabulario guardado en", vocab_path / "vocab.json")


### 4.2 Dataset y DataLoaders para la RNN

In [ ]:
class AGNewsSeqDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        ids, length = encode(self.texts[i], self.max_len)
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(self.labels[i], dtype=torch.long))

BATCH_SIZE_RNN = 128

train_ds = AGNewsSeqDataset(train_text, train_label)
val_ds   = AGNewsSeqDataset(val_text, val_label)
test_ds  = AGNewsSeqDataset(test_text, test_label)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_RNN, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print("Batches train:", len(train_loader))


### 4.3 Definición del modelo

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.context = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h, mask):
        # h: (B, L, H)  mask: (B, L) con 1 en tokens reales
        scores = self.context(torch.tanh(self.proj(h))).squeeze(-1)  # (B, L)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=1)                       # (B, L)
        pooled = torch.bmm(weights.unsqueeze(1), h).squeeze(1)       # (B, H)
        return pooled, weights

class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=200, hidden_dim=256,
                 num_classes=NUM_CLASSES, pad_idx=PAD_IDX, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.attn = AdditiveAttention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes),
        )
        self.pad_idx = pad_idx

    def forward(self, x, lengths):
        mask = (x != self.pad_idx).long()                  # (B, L)
        emb = self.embedding(x)                            # (B, L, E)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        pooled, _ = self.attn(out, mask)
        return self.fc(self.dropout(pooled))

set_seed()
rnn_model = BiLSTMAttentionClassifier(len(itos)).to(DEVICE)
n_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
print(rnn_model)
print(f"Parametros entrenables: {n_params:,}")


### 4.4 Bucle de entrenamiento con reanudación y checkpoint por época

In [ ]:
@torch.no_grad()
def evaluate_rnn(model, loader):
    model.eval()
    ys, ps, total_loss, n = [], [], 0.0, 0
    crit = nn.CrossEntropyLoss()
    for x, lengths, y in loader:
        x, lengths, y = x.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
        logits = model(x, lengths)
        total_loss += crit(logits, y).item() * y.size(0)
        n += y.size(0)
        ps.extend(logits.argmax(1).cpu().tolist())
        ys.extend(y.cpu().tolist())
    m = compute_metrics(ys, ps)
    m["loss"] = total_loss / n
    return m, ys, ps

def train_rnn(model, epochs=12, lr=1e-3, patience=3):
    ckpt_dir = CKPT_ROOT / "rnn_scratch"
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    start_epoch, best_f1, history = load_checkpoint(ckpt_dir, model, optimizer, scheduler)
    bad = 0

    for epoch in range(start_epoch + 1, epochs + 1):
        model.train()
        t0 = time.time()
        running = 0.0
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x, lengths), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * y.size(0)
        scheduler.step()
        train_loss = running / len(train_ds)
        val_m, _, _ = evaluate_rnn(model, val_loader)
        dt = time.time() - t0
        print(f"Epoca {epoch:02d}/{epochs} | train_loss {train_loss:.4f} | "
              f"val_loss {val_m['loss']:.4f} | val_acc {val_m['accuracy']:.4f} | "
              f"val_macroF1 {val_m['macro_f1']:.4f} | {dt:.1f}s")

        history.append({"epoch": epoch, "train_loss": train_loss, **val_m})
        is_best = val_m["macro_f1"] > best_f1
        if is_best:
            best_f1 = val_m["macro_f1"]; bad = 0
        else:
            bad += 1
        # >>> checkpoint DESPUES de cada epoca <<<
        save_checkpoint(ckpt_dir, model, optimizer, scheduler, epoch,
                        best_f1, history, is_best=is_best)
        if bad >= patience:
            print(f"Early stopping en epoca {epoch} (sin mejora {patience} epocas).")
            break
    return history

rnn_history = train_rnn(rnn_model, epochs=12)


### 4.5 Evaluación en test (carga del mejor checkpoint)

In [ ]:
def eval_best_rnn():
    ckpt_dir = CKPT_ROOT / "rnn_scratch"
    best = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
    rnn_model.load_state_dict(best["model_state"])
    m, ys, ps = evaluate_rnn(rnn_model, test_loader)
    print("== RNN desde cero · TEST ==")
    print(f"accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")
    print(classification_report(ys, ps, target_names=CLASS_NAMES, digits=4))
    return m

rnn_test = eval_best_rnn()


## 5. Modelo 2 — TF-IDF + Regresión Softmax

Línea base clásica en su **versión final**: se vectoriza el texto con TF-IDF (unigramas + bigramas) y
se entrena una **regresión logística multinomial** implementada como una capa lineal + softmax en
PyTorch, optimizada con AdamW. También aquí se **guarda un checkpoint por época** y se reanuda solo.

### 5.1 Vectorización TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
import joblib

tfidf = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2),
    min_df=5, max_features=50000, sublinear_tf=True,
)
X_train = tfidf.fit_transform(train_text)
X_val   = tfidf.transform(val_text)
X_test  = tfidf.transform(test_text)
print("TF-IDF train shape:", X_train.shape)

tfidf_dir = CKPT_ROOT / "tfidf_softmax"
tfidf_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf, tfidf_dir / "tfidf_vectorizer.joblib")
print("Vectorizador TF-IDF guardado en", tfidf_dir / "tfidf_vectorizer.joblib")


### 5.2 Dataset disperso y modelo softmax lineal

In [ ]:
class SparseDataset(Dataset):
    def __init__(self, X_csr: csr_matrix, y):
        self.X = X_csr.tocsr()
        self.y = np.asarray(y, dtype=np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        row = self.X[i]
        dense = torch.from_numpy(row.toarray().ravel().astype(np.float32))
        return dense, torch.tensor(self.y[i], dtype=torch.long)

BATCH_SIZE_TFIDF = 256
INPUT_DIM = X_train.shape[1]

tfidf_train_loader = DataLoader(SparseDataset(X_train, train_label),
                                batch_size=BATCH_SIZE_TFIDF, shuffle=True, num_workers=2)
tfidf_val_loader   = DataLoader(SparseDataset(X_val, val_label),
                                batch_size=512, shuffle=False, num_workers=2)
tfidf_test_loader  = DataLoader(SparseDataset(X_test, test_label),
                                batch_size=512, shuffle=False, num_workers=2)

class SoftmaxRegression(nn.Module):
    def __init__(self, input_dim, num_classes=NUM_CLASSES):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

set_seed()
tfidf_model = SoftmaxRegression(INPUT_DIM).to(DEVICE)
print(f"Dimension de entrada TF-IDF: {INPUT_DIM:,}")


### 5.3 Entrenamiento con checkpoint por época y reanudación

In [ ]:
@torch.no_grad()
def evaluate_tfidf(model, loader):
    model.eval()
    ys, ps, total_loss, n = [], [], 0.0, 0
    crit = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += crit(logits, y).item() * y.size(0)
        n += y.size(0)
        ps.extend(logits.argmax(1).cpu().tolist())
        ys.extend(y.cpu().tolist())
    m = compute_metrics(ys, ps)
    m["loss"] = total_loss / n
    return m, ys, ps

def train_tfidf(model, epochs=15, lr=1e-2, weight_decay=1e-4, patience=3):
    ckpt_dir = CKPT_ROOT / "tfidf_softmax"
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    start_epoch, best_f1, history = load_checkpoint(ckpt_dir, model, optimizer, scheduler)
    bad = 0

    for epoch in range(start_epoch + 1, epochs + 1):
        model.train()
        t0 = time.time(); running = 0.0
        for x, y in tfidf_train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running += loss.item() * y.size(0)
        scheduler.step()
        train_loss = running / len(train_text)
        val_m, _, _ = evaluate_tfidf(model, tfidf_val_loader)
        dt = time.time() - t0
        print(f"Epoca {epoch:02d}/{epochs} | train_loss {train_loss:.4f} | "
              f"val_loss {val_m['loss']:.4f} | val_acc {val_m['accuracy']:.4f} | "
              f"val_macroF1 {val_m['macro_f1']:.4f} | {dt:.1f}s")

        history.append({"epoch": epoch, "train_loss": train_loss, **val_m})
        is_best = val_m["macro_f1"] > best_f1
        if is_best:
            best_f1 = val_m["macro_f1"]; bad = 0
        else:
            bad += 1
        save_checkpoint(ckpt_dir, model, optimizer, scheduler, epoch,
                        best_f1, history, is_best=is_best)
        if bad >= patience:
            print(f"Early stopping en epoca {epoch}.")
            break
    return history

tfidf_history = train_tfidf(tfidf_model, epochs=15)


### 5.4 Evaluación en test

In [ ]:
def eval_best_tfidf():
    ckpt_dir = CKPT_ROOT / "tfidf_softmax"
    best = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
    tfidf_model.load_state_dict(best["model_state"])
    m, ys, ps = evaluate_tfidf(tfidf_model, tfidf_test_loader)
    print("== TF-IDF + Softmax · TEST ==")
    print(f"accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")
    print(classification_report(ys, ps, target_names=CLASS_NAMES, digits=4))
    return m

tfidf_test = eval_best_tfidf()


## 6. Modelo 3 — DistilBERT ajustado (fine-tuning)

Se parte de `distilbert-base-uncased`, se añade una cabeza de clasificación de 4 clases y se ajusta
toda la red. El entrenamiento usa **precisión mixta (AMP)** en GPU y, como los demás, guarda un
**checkpoint por época** con reanudación automática. Se guarda también el *tokenizer* para servir la API.

### 6.1 Tokenización y DataLoaders

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_NAME = "distilbert-base-uncased"
MAX_TOKENS = 128
BATCH_SIZE_BERT = 32

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AGNewsBertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_TOKENS):
        self.texts = texts; self.labels = labels
        self.tok = tokenizer; self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[i], dtype=torch.long),
        }

bert_train_loader = DataLoader(AGNewsBertDataset(train_text, train_label, bert_tokenizer),
                               batch_size=BATCH_SIZE_BERT, shuffle=True, num_workers=2, pin_memory=True)
bert_val_loader   = DataLoader(AGNewsBertDataset(val_text, val_label, bert_tokenizer),
                               batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
bert_test_loader  = DataLoader(AGNewsBertDataset(test_text, test_label, bert_tokenizer),
                               batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

bert_dir = CKPT_ROOT / "distilbert"
bert_dir.mkdir(parents=True, exist_ok=True)
bert_tokenizer.save_pretrained(bert_dir / "tokenizer")
print("Tokenizer guardado en", bert_dir / "tokenizer")


### 6.2 Modelo y entrenamiento con AMP, checkpoint por época y reanudación

In [ ]:
set_seed()
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES,
    id2label={i: c for i, c in enumerate(CLASS_NAMES)},
    label2id={c: i for i, c in enumerate(CLASS_NAMES)},
).to(DEVICE)

@torch.no_grad()
def evaluate_bert(model, loader):
    model.eval()
    ys, ps, total_loss, n = [], [], 0.0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        total_loss += out.loss.item() * labels.size(0); n += labels.size(0)
        ps.extend(out.logits.argmax(1).cpu().tolist())
        ys.extend(labels.cpu().tolist())
    m = compute_metrics(ys, ps); m["loss"] = total_loss / n
    return m, ys, ps

def train_bert(model, epochs=3, lr=2e-5, warmup_ratio=0.1, patience=2):
    ckpt_dir = CKPT_ROOT / "distilbert"
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps = len(bert_train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * warmup_ratio), total_steps)
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    start_epoch, best_f1, history = load_checkpoint(ckpt_dir, model, optimizer, scheduler)
    bad = 0

    for epoch in range(start_epoch + 1, epochs + 1):
        model.train(); t0 = time.time(); running = 0.0; seen = 0
        for batch in bert_train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
                loss = out.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item() * labels.size(0); seen += labels.size(0)
        train_loss = running / seen
        val_m, _, _ = evaluate_bert(model, bert_val_loader)
        dt = time.time() - t0
        print(f"Epoca {epoch:02d}/{epochs} | train_loss {train_loss:.4f} | "
              f"val_loss {val_m['loss']:.4f} | val_acc {val_m['accuracy']:.4f} | "
              f"val_macroF1 {val_m['macro_f1']:.4f} | {dt:.1f}s")

        history.append({"epoch": epoch, "train_loss": train_loss, **val_m})
        is_best = val_m["macro_f1"] > best_f1
        if is_best:
            best_f1 = val_m["macro_f1"]; bad = 0
        else:
            bad += 1
        save_checkpoint(ckpt_dir, model, optimizer, scheduler, epoch,
                        best_f1, history, is_best=is_best)
        # Guardar tambien en formato HuggingFace para servir facil la API.
        if is_best:
            model.save_pretrained(ckpt_dir / "hf_best")
        if bad >= patience:
            print(f"Early stopping en epoca {epoch}.")
            break
    return history

bert_history = train_bert(bert_model, epochs=3)


### 6.3 Evaluación en test

In [ ]:
def eval_best_bert():
    ckpt_dir = CKPT_ROOT / "distilbert"
    best = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
    bert_model.load_state_dict(best["model_state"])
    m, ys, ps = evaluate_bert(bert_model, bert_test_loader)
    print("== DistilBERT ajustado · TEST ==")
    print(f"accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")
    print(classification_report(ys, ps, target_names=CLASS_NAMES, digits=4))
    return m

bert_test = eval_best_bert()


## 7. Modelo 4 — NewsHybridNet (arquitectura propuesta)

Implementación fiel a la sección 5.3 del anteproyecto. Red de **doble rama** con fusión por compuerta:

- **Rama A — semántica secuencial:** embeddings fastText de 300d **congelados** (OOV compuesto por
  subpalabras) → adaptador `Linear 300→256 + LayerNorm + GELU + SpatialDropout(0.2)` → tres `Conv1D`
  paralelas (kernels 2, 3, 5; 128 filtros; padding "same") → concatenación + `LayerNorm + ReLU` (384) →
  `BiGRU` 128+128 (256) → **atención aditiva enmascarada ‖ max-pooling temporal** → `h_A` (512).
- **Rama B — léxica de subpalabras:** n-gramas de caracteres n = 3..5 con *hashing* a 2¹⁷ = 131.072
  cubetas → `EmbeddingBag` (131072×64, agregación por media) → `h_B` (64).
- **Fusión con compuerta:** proyecciones `h_A→256` y `h_B→256`; `g = σ(W·[h_A,h_B]+b)`;
  `h = g ⊙ h_A' + (1 − g) ⊙ h_B'`.
- **Cabeza:** `Dropout(0.3) + Dense 256→128 + GELU → Dense 128→4 → Softmax`.

Entrenamiento (sección 5.8): CE con *label smoothing* 0.1, AdamW `lr=1e-3`, `weight_decay=1e-2`
(sin decay en LayerNorm y sesgos), scheduler coseno con *warmup*, lote 128, 10–15 épocas con early
stopping (paciencia 3) y recorte de gradiente a 1.0. Igual que los otros modelos, **guarda checkpoint
por época y se reanuda automáticamente**.


### 7.1 Embeddings fastText congelados (con composición de subpalabras para OOV)

El anteproyecto exige embeddings fastText **congelados** y que las palabras fuera de vocabulario reciban
el vector que fastText compone a partir de sus subpalabras. Eso solo es posible con el modelo `.bin`
(formato Facebook), no con el `.vec`. La celda intenta, en este orden:

1. un `.bin` de fastText ya presente en Kaggle (añade el dataset *fasttext-english* a tu notebook), o
2. descargarlo con la librería `gensim` / `fasttext` si hay internet.

Si no se encuentra ninguno, se construye una matriz **aleatoria congelada** como respaldo para que el
notebook no se rompa, dejando un aviso explícito: en ese caso la rama A no usa conocimiento preentrenado
y el resultado no corresponde a la configuración propuesta. Define `FASTTEXT_BIN_PATH` si tu `.bin`
está en otra ruta.

In [ ]:
# Reutiliza el vocabulario por palabras (itos/stoi/MAX_LEN) construido para el modelo 1.
EMB_DIM = 300
FASTTEXT_BIN_PATH = None  # p.ej. "/kaggle/input/fasttext-english/wiki-news-300d-1M-subword.bin"

_CANDIDATE_BINS = [
    FASTTEXT_BIN_PATH,
    "/kaggle/input/fasttext-english/wiki-news-300d-1M-subword.bin",
    "/kaggle/input/fasttext-wikinews/wiki-news-300d-1M-subword.bin",
    "/kaggle/input/fasttext/cc.en.300.bin",
    "/kaggle/input/fasttextenglish/crawl-300d-2M-subword.bin",
]

def _load_fasttext_model():
    """Devuelve un objeto con metodo get_vector(word)->np.ndarray(300,) o None."""
    import numpy as np
    # 1) .bin local via libreria oficial fasttext (mejor manejo de subwords/OOV)
    for path in _CANDIDATE_BINS:
        if path and os.path.exists(path):
            try:
                import fasttext
                ft = fasttext.load_model(path)
                print("fastText (.bin) cargado con libreria 'fasttext':", path)
                class _W:
                    def get_vector(self, w): return ft.get_word_vector(w)
                return _W()
            except Exception as e:
                print("No se pudo cargar con 'fasttext' (", e, "); intento gensim...")
                try:
                    from gensim.models.fasttext import load_facebook_vectors
                    kv = load_facebook_vectors(path)
                    print("fastText (.bin) cargado con gensim:", path)
                    class _W:
                        def get_vector(self, w):
                            try: return kv.get_vector(w)
                            except Exception: return np.zeros(EMB_DIM, dtype=np.float32)
                    return _W()
                except Exception as e2:
                    print("gensim tambien fallo:", e2)
    # 2) Descargar con gensim.downloader (requiere internet). Nota: la version .vec de
    #    gensim-data NO compone OOV, pero sirve como aproximacion si no hay .bin.
    try:
        import gensim.downloader as api
        print("Descargando 'fasttext-wiki-news-subwords-300' via gensim.downloader (puede tardar)...")
        kv = api.load("fasttext-wiki-news-subwords-300")
        class _W:
            def get_vector(self, w):
                try: return kv.get_vector(w)
                except Exception: return np.zeros(EMB_DIM, dtype=np.float32)
        return _W()
    except Exception as e:
        print("No se pudo descargar via gensim.downloader:", e)
    return None

def build_embedding_matrix(itos, dim=EMB_DIM):
    import numpy as np
    ft = _load_fasttext_model()
    matrix = np.random.normal(0.0, 0.1, size=(len(itos), dim)).astype(np.float32)
    matrix[PAD_IDX] = 0.0
    if ft is None:
        print("\n*** AVISO: no se encontro fastText. Se usa matriz ALEATORIA congelada. ***")
        print("*** Esto NO corresponde a la configuracion propuesta (rama A sin preentrenamiento). ***")
        return matrix, False
    hits = 0
    for i, tok in enumerate(itos):
        if tok in ("<pad>", "<unk>"):
            continue
        try:
            vec = ft.get_vector(tok)  # fastText compone OOV por subpalabras
            if vec is not None and np.linalg.norm(vec) > 0:
                matrix[i] = vec[:dim]; hits += 1
        except Exception:
            pass
    print(f"Vectores fastText asignados: {hits}/{len(itos)} ({100*hits/len(itos):.1f}%)")
    return matrix, True

emb_matrix, USED_FASTTEXT = build_embedding_matrix(itos)
emb_tensor = torch.tensor(emb_matrix)
print("Matriz de embeddings:", tuple(emb_tensor.shape), "| fastText:", USED_FASTTEXT)


### 7.2 Preprocesamiento de la rama B — n-gramas de caracteres con *hashing*

Para cada texto se extraen n-gramas de caracteres (n = 3, 4, 5) sobre el texto normalizado y se mapean
a cubetas mediante *hashing* a 2¹⁷ = 131.072, tal como indica el anteproyecto. El `Dataset` produce,
por ejemplo, la secuencia de palabras (para la rama A) y la bolsa de índices de n-gramas (para la rama B).

In [ ]:
NGRAM_BUCKETS = 2 ** 17          # 131072 cubetas de hashing
CHAR_NGRAMS = (3, 4, 5)

_norm_re = re.compile(r"\s+")

def _normalize_for_ngrams(text):
    return _norm_re.sub(" ", text.lower()).strip()

def char_ngram_hashes(text, buckets=NGRAM_BUCKETS, ns=CHAR_NGRAMS):
    s = "<" + _normalize_for_ngrams(text) + ">"
    idxs = []
    for n in ns:
        if len(s) < n:
            continue
        for i in range(len(s) - n + 1):
            gram = s[i:i + n]
            idxs.append(hash(gram) % buckets)
    if not idxs:
        idxs = [0]
    return idxs

class AGNewsHybridDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.texts = texts; self.labels = labels; self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        ids, length = encode(self.texts[i], self.max_len)      # rama A (palabras)
        ngrams = char_ngram_hashes(self.texts[i])              # rama B (n-gramas char)
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(ngrams, dtype=torch.long),
                torch.tensor(self.labels[i], dtype=torch.long))

def hybrid_collate(batch):
    """Empaqueta la bolsa de n-gramas en el formato de EmbeddingBag (valores + offsets)."""
    ids = torch.stack([b[0] for b in batch])
    lengths = torch.stack([b[1] for b in batch])
    labels = torch.stack([b[3] for b in batch])
    ngram_lists = [b[2] for b in batch]
    offsets = torch.zeros(len(ngram_lists), dtype=torch.long)
    acc = 0
    for j, g in enumerate(ngram_lists):
        offsets[j] = acc; acc += g.numel()
    values = torch.cat(ngram_lists) if ngram_lists else torch.zeros(1, dtype=torch.long)
    return ids, lengths, values, offsets, labels

BATCH_SIZE_HYBRID = 128
hyb_train_loader = DataLoader(AGNewsHybridDataset(train_text, train_label),
                              batch_size=BATCH_SIZE_HYBRID, shuffle=True,
                              num_workers=2, pin_memory=True, collate_fn=hybrid_collate)
hyb_val_loader   = DataLoader(AGNewsHybridDataset(val_text, val_label),
                              batch_size=256, shuffle=False,
                              num_workers=2, pin_memory=True, collate_fn=hybrid_collate)
hyb_test_loader  = DataLoader(AGNewsHybridDataset(test_text, test_label),
                              batch_size=256, shuffle=False,
                              num_workers=2, pin_memory=True, collate_fn=hybrid_collate)
print("Batches train (hybrid):", len(hyb_train_loader))


### 7.3 Definición de NewsHybridNet (capa por capa, según la tabla 5.3.2)

In [ ]:
class SpatialDropout1d(nn.Module):
    """Dropout que elimina canales completos a lo largo del tiempo (B, L, C)."""
    def __init__(self, p):
        super().__init__(); self.drop = nn.Dropout2d(p)

    def forward(self, x):                 # x: (B, L, C)
        x = x.permute(0, 2, 1).unsqueeze(-1)   # (B, C, L, 1)
        x = self.drop(x)
        return x.squeeze(-1).permute(0, 2, 1)  # (B, L, C)

class MaskedAdditiveAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.context = nn.Linear(dim, 1, bias=False)

    def forward(self, h, mask):           # h: (B, L, D)  mask: (B, L)
        scores = self.context(torch.tanh(self.proj(h))).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=1)
        pooled = torch.bmm(w.unsqueeze(1), h).squeeze(1)
        return pooled, w

class NewsHybridNet(nn.Module):
    def __init__(self, emb_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX,
                 ngram_buckets=NGRAM_BUCKETS):
        super().__init__()
        vocab_size, emb_dim = emb_matrix.shape
        self.pad_idx = pad_idx

        # --- Rama A: semantica secuencial ---
        # A1: embeddings fastText CONGELADOS
        self.embedding = nn.Embedding.from_pretrained(
            emb_matrix, freeze=True, padding_idx=pad_idx)
        # A2: adaptador 300 -> 256 + LayerNorm + GELU + spatial dropout 0.2
        self.adapter = nn.Linear(emb_dim, 256)
        self.adapter_ln = nn.LayerNorm(256)
        self.spatial_drop = SpatialDropout1d(0.2)
        # A3: tres Conv1D paralelas (k=2,3,5; 128 filtros; padding "same")
        self.conv2 = nn.Conv1d(256, 128, kernel_size=2, padding="same")
        self.conv3 = nn.Conv1d(256, 128, kernel_size=3, padding="same")
        self.conv5 = nn.Conv1d(256, 128, kernel_size=5, padding="same")
        # A4: concat (384) + LayerNorm + ReLU
        self.cat_ln = nn.LayerNorm(384)
        # A5: BiGRU 128+128 -> 256
        self.bigru = nn.GRU(384, 128, num_layers=1, batch_first=True, bidirectional=True)
        # A6: atencion aditiva enmascarada || max-pooling temporal -> 512
        self.attn = MaskedAdditiveAttention(256)

        # --- Rama B: lexica de subpalabras ---
        self.embbag = nn.EmbeddingBag(ngram_buckets, 64, mode="mean")

        # --- Fusion con compuerta ---
        self.proj_a = nn.Linear(512, 256)   # h_A -> 256
        self.proj_b = nn.Linear(64, 256)    # h_B -> 256
        self.gate = nn.Linear(512, 256)     # g = sigma(W[h_A',h_B'] + b)

        # --- Cabeza ---
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(256, 128), nn.GELU(), nn.Linear(128, num_classes),
        )

    def forward(self, x, lengths, ngram_values, ngram_offsets, return_gate=False):
        mask = (x != self.pad_idx).long()                 # (B, L)
        # Rama A
        emb = self.embedding(x)                           # (B, L, 300)
        a = F.gelu(self.adapter_ln(self.adapter(emb)))    # (B, L, 256)
        a = self.spatial_drop(a)
        a_t = a.transpose(1, 2)                           # (B, 256, L)
        c = torch.cat([self.conv2(a_t), self.conv3(a_t), self.conv5(a_t)], dim=1)  # (B,384,L)
        c = c.transpose(1, 2)                             # (B, L, 384)
        c = F.relu(self.cat_ln(c))
        packed = nn.utils.rnn.pack_padded_sequence(
            c, lengths.cpu(), batch_first=True, enforce_sorted=False)
        g_out, _ = self.bigru(packed)
        g_out, _ = nn.utils.rnn.pad_packed_sequence(
            g_out, batch_first=True, total_length=x.size(1))   # (B, L, 256)
        att, _ = self.attn(g_out, mask)                   # (B, 256)
        masked = g_out.masked_fill((mask == 0).unsqueeze(-1), float("-inf"))
        maxp, _ = masked.max(dim=1)                       # (B, 256)
        maxp = torch.nan_to_num(maxp, neginf=0.0)
        h_a = torch.cat([att, maxp], dim=1)               # (B, 512)

        # Rama B
        h_b = self.embbag(ngram_values, ngram_offsets)    # (B, 64)

        # Fusion con compuerta
        ha_p = self.proj_a(h_a)                           # (B, 256)
        hb_p = self.proj_b(h_b)                           # (B, 256)
        g = torch.sigmoid(self.gate(torch.cat([ha_p, hb_p], dim=1)))  # (B, 256)
        fused = g * ha_p + (1 - g) * hb_p                 # (B, 256)

        logits = self.head(fused)
        if return_gate:
            return logits, g
        return logits

set_seed()
hybrid_model = NewsHybridNet(emb_tensor).to(DEVICE)
total_p = sum(p.numel() for p in hybrid_model.parameters())
train_p = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
print(hybrid_model)
print(f"Parametros totales: {total_p:,} | entrenables: {train_p:,} | congelados: {total_p-train_p:,}")


### 7.4 Entrenamiento con checkpoint por época y reanudación

Se excluyen `LayerNorm` y sesgos del *weight decay* (como pide la sección 5.8) y se guardan, además del
checkpoint, los artefactos para servir la API: vocabulario, matriz de embeddings y la config de n-gramas.

In [ ]:
def build_param_groups(model, weight_decay=1e-2):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or name.endswith(".bias") or "ln" in name.lower() or "norm" in name.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]

@torch.no_grad()
def evaluate_hybrid(model, loader):
    model.eval()
    ys, ps, total_loss, n = [], [], 0.0, 0
    crit = nn.CrossEntropyLoss()
    for ids, lengths, vals, offs, y in loader:
        ids, lengths = ids.to(DEVICE), lengths.to(DEVICE)
        vals, offs, y = vals.to(DEVICE), offs.to(DEVICE), y.to(DEVICE)
        logits = model(ids, lengths, vals, offs)
        total_loss += crit(logits, y).item() * y.size(0); n += y.size(0)
        ps.extend(logits.argmax(1).cpu().tolist()); ys.extend(y.cpu().tolist())
    m = compute_metrics(ys, ps); m["loss"] = total_loss / n
    return m, ys, ps

def save_hybrid_artifacts(ckpt_dir):
    import numpy as np
    np.save(ckpt_dir / "embeddings.npy", emb_matrix)
    with open(ckpt_dir / "vocab.json", "w") as f:
        json.dump({"itos": itos, "max_len": MAX_LEN, "pad_idx": PAD_IDX}, f)
    with open(ckpt_dir / "ngram_config.json", "w") as f:
        json.dump({"buckets": NGRAM_BUCKETS, "ns": list(CHAR_NGRAMS)}, f)

def train_hybrid(model, epochs=12, lr=1e-3, patience=3):
    ckpt_dir = CKPT_ROOT / "newshybridnet"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_hybrid_artifacts(ckpt_dir)

    optimizer = torch.optim.AdamW(build_param_groups(model, 1e-2), lr=lr)
    steps_per_epoch = len(hyb_train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps_per_epoch, pct_start=0.1)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    start_epoch, best_f1, history = load_checkpoint(ckpt_dir, model, optimizer, scheduler)
    bad = 0
    for epoch in range(start_epoch + 1, epochs + 1):
        model.train(); t0 = time.time(); running = 0.0
        for ids, lengths, vals, offs, y in hyb_train_loader:
            ids, lengths = ids.to(DEVICE), lengths.to(DEVICE)
            vals, offs, y = vals.to(DEVICE), offs.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(ids, lengths, vals, offs), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += loss.item() * y.size(0)
        train_loss = running / len(train_text)
        val_m, _, _ = evaluate_hybrid(model, hyb_val_loader)
        dt = time.time() - t0
        print(f"Epoca {epoch:02d}/{epochs} | train_loss {train_loss:.4f} | "
              f"val_loss {val_m['loss']:.4f} | val_acc {val_m['accuracy']:.4f} | "
              f"val_macroF1 {val_m['macro_f1']:.4f} | {dt:.1f}s")
        history.append({"epoch": epoch, "train_loss": train_loss, **val_m})
        is_best = val_m["macro_f1"] > best_f1
        if is_best:
            best_f1 = val_m["macro_f1"]; bad = 0
        else:
            bad += 1
        save_checkpoint(ckpt_dir, model, optimizer, scheduler, epoch,
                        best_f1, history, extra={"used_fasttext": USED_FASTTEXT}, is_best=is_best)
        if bad >= patience:
            print(f"Early stopping en epoca {epoch}.")
            break
    return history

hybrid_history = train_hybrid(hybrid_model, epochs=12)


### 7.5 Evaluación en test

In [ ]:
def eval_best_hybrid():
    ckpt_dir = CKPT_ROOT / "newshybridnet"
    best = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
    hybrid_model.load_state_dict(best["model_state"])
    m, ys, ps = evaluate_hybrid(hybrid_model, hyb_test_loader)
    print("== NewsHybridNet · TEST ==")
    print(f"accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")
    print(classification_report(ys, ps, target_names=CLASS_NAMES, digits=4))
    return m

hybrid_test = eval_best_hybrid()


## 8. Resumen comparativo y artefactos guardados

Tabla final con accuracy y macro-F1 en test de los cuatro modelos, y un listado de todo lo que quedó
guardado en `checkpoints/` para servir luego la API.

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"modelo": "RNN desde cero (BiLSTM+attn)", **rnn_test},
    {"modelo": "TF-IDF + Softmax",             **tfidf_test},
    {"modelo": "DistilBERT ajustado",          **bert_test},
    {"modelo": "NewsHybridNet",                **hybrid_test},
])[["modelo", "accuracy", "macro_f1"]]
summary.to_csv(OUTPUT_DIR / "resumen_resultados.csv", index=False)
print(summary.to_string(index=False))

print("\n== Artefactos guardados ==")
for p in sorted(CKPT_ROOT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1e6:.2f} MB)")


## 9. Notas sobre los checkpoints y la reanudación

- Cada `train_*` llama a `load_checkpoint` al inicio. Si Kaggle corta la sesión a mitad del entrenamiento,
  basta con **volver a ejecutar la celda** (o el notebook completo): continuará desde la última época guardada.
- `last.pt` se escribe de forma **atómica** tras cada época, así que un corte durante el guardado no corrompe el archivo.
- `best.pt` guarda el mejor modelo por macro-F1 de validación; es el que se usa para evaluar en test y el que debe cargar la API.
- Para DistilBERT se guarda además `hf_best/` en formato Hugging Face (`from_pretrained`) para servir la API sin reconstruir la arquitectura a mano.
- Para **NewsHybridNet** se guardan `vocab.json`, `embeddings.npy` y `ngram_config.json` junto al checkpoint, de modo que la API pueda reconstruir el modelo y reproducir exactamente el preprocesamiento de ambas ramas.
- Todo vive bajo `/kaggle/working/checkpoints`, que Kaggle conserva como *output* del notebook y puedes descargar o versionar como *dataset*.
